In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
}

In [ ]:
data_gdynia = pd.read_csv(
    "../data/processed/fixed/srodmiescie_raw_object_id.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)
data_gdynia

In [ ]:
data_gdynia_long = pd.read_csv(
    "../data/processed/fixed/srodmiescie_long.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)
data_gdynia_long.head()

1.

In [ ]:
def storeys_range(s):
    return s.max() - s.min()


data_gdynia_starowiejska_num_storeys = data_gdynia.loc[
    data_gdynia["addr_street"] == "Starowiejska", "number_of_storeys"
]
storeys_range(data_gdynia_starowiejska_num_storeys)

In [ ]:
data_dynia_max_min_func = data_gdynia.groupby('addr_street')["number_of_storeys"].agg(storeys_range)

data_gdynia_max_min_diff = data_gdynia.groupby("addr_street").agg(
    max_storeys=("number_of_storeys", "max"), min_storeys=("number_of_storeys", "min")
).assign(storeys_range=lambda d: d["max_storeys"] - d["min_storeys"])

data_gdynia_max_min_lambda = data_gdynia.groupby("addr_street")["number_of_storeys"].agg(
    lambda s: s.max() - s.min())

In [ ]:
assert (data_dynia_max_min_func).equals(data_gdynia_max_min_diff.loc[:, 'storeys_range'])
assert (data_dynia_max_min_func).equals(data_gdynia_max_min_lambda)

2.

In [ ]:
def diff_from_median(s):
    return s - s.median()

print(len(diff_from_median(data_gdynia_starowiejska_num_storeys)))
diff_from_median(data_gdynia_starowiejska_num_storeys).head()


In [ ]:
data_gdynia_diff_median = data_gdynia.assign(storeys_vs_median = data_gdynia.groupby('addr_street')['number_of_storeys'].transform(diff_from_median))
data_gdynia_diff_median = data_gdynia_diff_median.loc[:,['addr_street','number_of_storeys','storeys_vs_median' ]]

In [ ]:
assert np.isclose(data_gdynia_diff_median.groupby('addr_street')['storeys_vs_median'].agg('median'), 0).all()

3.

In [ ]:
def share_tall(s, threshold):
    return (s >= threshold).mean()

share_tall(data_gdynia_starowiejska_num_storeys, 5)

In [ ]:
data_gdynia_shares_threshold = data_gdynia.groupby('addr_street').agg(
    share_5 = ('number_of_storeys', lambda d: share_tall(d,5)),
    share_8 = ('number_of_storeys', lambda d: share_tall(d,8)),
)
data_gdynia_shares_threshold.head(10)

In [ ]:
assert len(data_gdynia_shares_threshold) == data_gdynia['addr_street'].nunique()
assert (data_gdynia_shares_threshold['share_5'] >= data_gdynia_shares_threshold['share_8']).all()

In [ ]:
data_gdynia_threshold_5 = data_gdynia.groupby('addr_street')['number_of_storeys'].agg(share_tall, 5)
data_gdynia_threshold_8 = data_gdynia.groupby('addr_street')['number_of_storeys'].agg(share_tall, 8)

In [ ]:
pd.DataFrame({"share_5": data_gdynia_threshold_5, "share_8": data_gdynia_threshold_8}).head(10)

4.

In [ ]:
table_starowiejska = data_gdynia.groupby('addr_street').get_group('Starowiejska')

In [ ]:
def street_profile(group):
      tallest_id = group['number_of_storeys'].idxmax()
      return pd.Series([len(group),group['number_of_storeys'].loc[tallest_id], group['arch_style'].loc[tallest_id]],index = ['num_build', 'max_storeys', 'tallest_style'])

street_profile(table_starowiejska)


In [ ]:
data_gdynia_street_profile = data_gdynia.groupby('addr_street').apply(street_profile)

In [ ]:
assert len(data_gdynia_street_profile) == data_gdynia['addr_street'].nunique()

assert (data_gdynia_street_profile['num_build']).equals(data_gdynia.groupby('addr_street')['osm_id'].size())

assert data_gdynia_street_profile['num_build'].sum() == len(data_gdynia) - data_gdynia['addr_street'].isna().sum()

assert ((data_gdynia_street_profile['max_storeys']) ==(data_gdynia.groupby('addr_street')['number_of_storeys'].agg('max'))).all()

5.

In [ ]:
def only_big_streets(df, min_build):
    return df.loc[df.groupby('addr_street').transform('size') >= min_build]

data_gdynia_big_streets_20 = data_gdynia.pipe(only_big_streets, min_build = 20)
data_gdynia_big_streets_10 = data_gdynia.pipe(only_big_streets, min_build = 10)

In [ ]:
assert data_gdynia_big_streets_20.equals(only_big_streets(data_gdynia, 20))
assert len(data_gdynia_big_streets_20) == 605
assert data_gdynia_big_streets_20['addr_street'].nunique() == 13
assert len(data_gdynia_big_streets_10) == 754
assert data_gdynia_big_streets_10['addr_street'].nunique() == 25
assert data_gdynia_big_streets_10['addr_street'].isna().sum() == 0

In [ ]:
data_gdynia.pipe(only_big_streets, min_build = 20).groupby('addr_street')['number_of_storeys'].agg(storeys_range).sort_values(ascending = False)